# Full pipeline

Data prep -> train base/sft_only/distilled -> evaluate -> layer analysis, all in one session so nothing has to persist across separate notebook runtimes.

Enable GPU first (Kaggle: Accelerator -> GPU T4 x2, Internet -> On). Rough timing on T4 x2: data prep ~2 min, training ~40 min total (base ~3, sft_only ~11, distilled ~25), evaluation ~1-2 h (a start/end line per condition), layer analysis ~20 min. The guard cell after training stops the run early if a trained model does not emit clean tool calls.


In [ ]:
import os
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q -r requirements-colab.txt

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')


import subprocess


def run_module(*args):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and
    raise if it fails (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, "-m", *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


In [ ]:
# re-sync to latest fix without re-cloning
!cd {REPO_DIR} && git checkout -- . && git pull

## 1. Data

In [ ]:
run_module("adbench.data.prepare", "--config", "configs/data.yaml")

In [ ]:
run_module("adbench.data.general_eval", "--config", "configs/experiment.yaml")

## 2. Training

In [ ]:
run_module("adbench.training.train", "--condition", "base")

In [ ]:
run_module("adbench.training.train", "--condition", "sft_only")

In [ ]:
run_module("adbench.training.train", "--condition", "distilled")

Guard: stop the run now if a trained model does not produce clean tool calls, before the long evaluation.

In [ ]:
for condition in ["sft_only", "distilled"]:
    run_module("adbench.evaluation.diagnose", "--condition", condition,
               "--skip-training-target", "--n", "5", "--require-tool-call", "--max-failures", "1")

In [ ]:
import matplotlib.pyplot as plt

from adbench.data.prepare import read_jsonl
from adbench.training.train import loss_log_path

sft_log = read_jsonl(loss_log_path("sft_only"))
distilled_log = read_jsonl(loss_log_path("distilled"))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot([r["step"] for r in sft_log], [r["sft_loss"] for r in sft_log], label="sft_only: sft_loss")
axes[0].plot([r["step"] for r in distilled_log], [r["sft_loss"] for r in distilled_log], label="distilled: sft_loss")
axes[0].set_title("SFT loss")
axes[0].legend()

axes[1].plot([r["step"] for r in distilled_log], [r["kd_loss"] for r in distilled_log], label="distilled: kd_loss", color="darkorange")
axes[1].set_title("KD loss")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Evaluation

In [ ]:
from adbench.evaluation.run_eval import evaluate_condition, write_eval_outputs
from adbench.training.train import CONDITIONS, REPO_ROOT, load_experiment_config

experiment_config = load_experiment_config("configs/experiment.yaml")
models_config = load_experiment_config("configs/models.yaml")
chain_lengths = experiment_config["harness"]["chain_lengths"]

all_rows = []
perplexities = {}
for condition in CONDITIONS:
    print(f"=== Evaluating {condition} ===")
    rows, perplexity = evaluate_condition(condition, experiment_config, models_config, chain_lengths)
    all_rows.extend(rows)
    perplexities[condition] = perplexity
    print(f"{condition}: {len(rows)} tasks run, perplexity={perplexity:.2f}")
    # Written after every condition so a crash later keeps what already finished.
    output = write_eval_outputs(all_rows, perplexities, REPO_ROOT / "results")

print("wrote results/eval_results.jsonl, eval_results.csv, eval_summary.json")

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(output["summary"])
summary_df[["condition", "chain_length", "n_tasks", "full_chain_success_rate",
            "per_step_success_rate", "clean_step_success_rate", "recovery_rate", "perplexity"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for condition in CONDITIONS:
    sub = summary_df[summary_df["condition"] == condition].sort_values("chain_length")
    ax.plot(sub["chain_length"], sub["full_chain_success_rate"], marker="o", label=condition)
ax.set_xlabel("chain length")
ax.set_ylabel("full-chain success rate")
ax.set_xticks(chain_lengths)
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from adbench.evaluation.metrics import error_category_breakdown

for condition in CONDITIONS:
    condition_rows = [r for r in all_rows if r["condition"] == condition]
    print(condition, error_category_breakdown(condition_rows))

## 4. Layer analysis

Teacher and each student condition are extracted in their own fresh process. Unsloth patches transformers globally on import, which breaks student hooks if it was already imported anywhere in this kernel.

In [ ]:
run_module("adbench.analysis.layer_analysis", "--role", "teacher")

In [ ]:
for condition in ["base", "sft_only", "distilled"]:
    run_module("adbench.analysis.layer_analysis", "--role", "student", "--condition", condition)

Compare + plot

In [ ]:
run_module("adbench.analysis.layer_analysis", "--role", "compare")

In [ ]:
from adbench.analysis.layer_analysis import read_layer_analysis_results

la_config = experiment_config["layer_analysis"]
df = pd.DataFrame(read_layer_analysis_results(REPO_ROOT / la_config["results_path"]))
streams = ["attention", "ffn"]
metrics = la_config["divergence_metrics"]
colors = {"base": "tab:gray", "sft_only": "tab:blue", "distilled": "tab:red"}

for input_set in ["tool_use", "general"]:
    fig, axes = plt.subplots(len(metrics), len(streams), figsize=(11, 4 * len(metrics)), squeeze=False)
    for row_idx, metric in enumerate(metrics):
        for col_idx, stream in enumerate(streams):
            ax = axes[row_idx][col_idx]
            for condition, color in colors.items():
                sub = df[(df["stream"] == stream) & (df["input_set"] == input_set) & (df["condition"] == condition)].sort_values("source_layer")
                ax.plot(sub["source_layer"], sub[metric], marker="o", label=condition, color=color)
            ax.set_title(f"{input_set}: {stream} - {metric}")
            ax.legend()
    plt.tight_layout()
    plt.show()